### Célula 1 — Carregando as tabelas Gold

In [0]:
import pyspark.sql.functions as F
import builtins

GOLD_PATH = "/Volumes/workspace/default/raw/gold/"

# Carregando todas as tabelas
f_tickets    = spark.read.format("delta").load(f"{GOLD_PATH}f_customer_support_tickets/")
dim_customer = spark.read.format("delta").load(f"{GOLD_PATH}dim_customer/")
dim_product  = spark.read.format("delta").load(f"{GOLD_PATH}dim_product/")
dim_type     = spark.read.format("delta").load(f"{GOLD_PATH}dim_type/")
dim_subject  = spark.read.format("delta").load(f"{GOLD_PATH}dim_subject/")
dim_status   = spark.read.format("delta").load(f"{GOLD_PATH}dim_status/")
dim_priority = spark.read.format("delta").load(f"{GOLD_PATH}dim_priority/")
dim_channel  = spark.read.format("delta").load(f"{GOLD_PATH}dim_channel/")
dim_calendario = spark.read.format("delta").load(f"{GOLD_PATH}dim_calendario/")

print("✅ Tabelas Gold carregadas!")
print(f"Fato: {f_tickets.count():,} registros")

✅ Tabelas Gold carregadas!
Fato: 8,469 registros


### Célula 2 — Testes da tabela fato

In [0]:
print("=" * 60)
print("DATA QUALITY CHECKS — TABELA FATO")
print("=" * 60)

erros = []

# Teste 1 — Volume
total = f_tickets.count()
if total == 8469:
    print(f"✅ Teste 1 — Volume: {total:,} registros")
else:
    erros.append(f"🔴 Teste 1 — Volume incorreto: {total:,}")
    print(erros[-1])

# Teste 2 — Ticket_ID sem nulos
nulos_id = f_tickets.filter(F.col("Ticket_ID").isNull()).count()
if nulos_id == 0:
    print(f"✅ Teste 2 — Ticket_ID sem nulos")
else:
    erros.append(f"🔴 Teste 2 — Ticket_ID com nulos: {nulos_id}")
    print(erros[-1])

# Teste 3 — Ticket_ID único
unicos = f_tickets.select("Ticket_ID").distinct().count()
if unicos == total:
    print(f"✅ Teste 3 — Ticket_ID único: {unicos:,}")
else:
    erros.append(f"🔴 Teste 3 — Duplicatas: {total - unicos}")
    print(erros[-1])

# Teste 4 — FKs sem nulos
fks = ["Customer_ID", "Product_ID", "Type_ID",
       "Subject_ID", "Status_ID", "Priority_ID", "Channel_ID"]
for fk in fks:
    nulos = f_tickets.filter(F.col(fk).isNull()).count()
    if nulos == 0:
        print(f"✅ Teste 4 — {fk} sem nulos")
    else:
        erros.append(f"🔴 Teste 4 — {fk} com nulos: {nulos}")
        print(erros[-1])

# Teste 5 — Is_Resolved só tem 0 ou 1
valores_invalidos = f_tickets.filter(
    ~F.col("Is_Resolved").isin([0, 1])
).count()
if valores_invalidos == 0:
    print(f"✅ Teste 5 — Is_Resolved só contém 0 e 1")
else:
    erros.append(f"🔴 Teste 5 — Is_Resolved com valores inválidos: {valores_invalidos}")
    print(erros[-1])

# Teste 6 — Satisfaction no range 1-5
sat_anomalias = f_tickets.filter(
    F.col("Customer_Satisfaction_Rating").isNotNull() &
    ((F.col("Customer_Satisfaction_Rating") < 1) |
     (F.col("Customer_Satisfaction_Rating") > 5))
).count()
if sat_anomalias == 0:
    print(f"✅ Teste 6 — Satisfaction no range (1-5)")
else:
    erros.append(f"🔴 Teste 6 — Satisfaction anomalias: {sat_anomalias}")
    print(erros[-1])

# Teste 7 — Resolution_Time_Hours não negativo
tempo_negativo = f_tickets.filter(
    F.col("Resolution_Time_Hours").isNotNull() &
    (F.col("Resolution_Time_Hours") < 0)
).count()
if tempo_negativo == 0:
    print(f"✅ Teste 7 — Resolution_Time_Hours sem valores negativos")
else:
    erros.append(f"🔴 Teste 7 — Tempo negativo: {tempo_negativo}")
    print(erros[-1])

# Teste 8 — Date_of_Purchase no range esperado
datas_invalidas = f_tickets.filter(
    (F.col("Date_of_Purchase") < F.lit("2020-01-01")) |
    (F.col("Date_of_Purchase") > F.lit("2021-12-31"))
).count()
if datas_invalidas == 0:
    print(f"✅ Teste 8 — Date_of_Purchase no range (2020-2021)")
else:
    erros.append(f"🔴 Teste 8 — Datas fora do range: {datas_invalidas}")
    print(erros[-1])

print()
print("=" * 60)
if len(erros) == 0:
    print("🎉 TODOS OS TESTES DA FATO PASSARAM!")
else:
    print(f"🔴 {len(erros)} TESTE(S) FALHARAM!")
    for e in erros: print(f"   {e}")
print("=" * 60)

DATA QUALITY CHECKS — TABELA FATO
✅ Teste 1 — Volume: 8,469 registros
✅ Teste 2 — Ticket_ID sem nulos
✅ Teste 3 — Ticket_ID único: 8,469
✅ Teste 4 — Customer_ID sem nulos
✅ Teste 4 — Product_ID sem nulos
✅ Teste 4 — Type_ID sem nulos
✅ Teste 4 — Subject_ID sem nulos
✅ Teste 4 — Status_ID sem nulos
✅ Teste 4 — Priority_ID sem nulos
✅ Teste 4 — Channel_ID sem nulos
✅ Teste 5 — Is_Resolved só contém 0 e 1
✅ Teste 6 — Satisfaction no range (1-5)
✅ Teste 7 — Resolution_Time_Hours sem valores negativos
✅ Teste 8 — Date_of_Purchase no range (2020-2021)

🎉 TODOS OS TESTES DA FATO PASSARAM!


### Célula 3 — Testes das dimensões

In [0]:
print("=" * 60)
print("DATA QUALITY CHECKS — DIMENSÕES")
print("=" * 60)

erros = []

# Teste 1 — Cardinalidade esperada
dimensoes_esperadas = {
    "dim_product"  : (dim_product,   42),
    "dim_type"     : (dim_type,       5),
    "dim_subject"  : (dim_subject,   16),
    "dim_status"   : (dim_status,     3),
    "dim_priority" : (dim_priority,   4),
    "dim_channel"  : (dim_channel,    4),
    "dim_customer" : (dim_customer, 8320),
    "dim_calendario": (dim_calendario, 730)
}

for nome, (df, esperado) in dimensoes_esperadas.items():
    total = df.count()
    if total == esperado:
        print(f"✅ {nome}: {total:,} registros")
    else:
        erros.append(f"🔴 {nome}: {total:,} (esperado: {esperado:,})")
        print(erros[-1])

print()

# Teste 2 — Integridade referencial — FKs da fato existem nas dimensões
print("─" * 60)
print("Integridade referencial — FKs x Dimensões")
print("─" * 60)

joins = [
    ("Customer_ID", dim_customer,  "Customer_ID",  "dim_customer"),
    ("Product_ID",  dim_product,   "Product_ID",   "dim_product"),
    ("Type_ID",     dim_type,      "Type_ID",       "dim_type"),
    ("Subject_ID",  dim_subject,   "Subject_ID",    "dim_subject"),
    ("Status_ID",   dim_status,    "Status_ID",     "dim_status"),
    ("Priority_ID", dim_priority,  "Priority_ID",   "dim_priority"),
    ("Channel_ID",  dim_channel,   "Channel_ID",    "dim_channel"),
]

for fk, dim, dim_key, dim_nome in joins:
    orphans = f_tickets.join(
        dim, f_tickets[fk] == dim[dim_key], "left_anti"             # registros sem match
    ).count()
    if orphans == 0:
        print(f"✅ {fk} → {dim_nome}: sem órfãos")
    else:
        erros.append(f"🔴 {fk} → {dim_nome}: {orphans} órfãos!")
        print(erros[-1])

print()

# Teste 3 — dim_calendario cobre todo o range da fato
print("─" * 60)
print("Cobertura do calendário")
print("─" * 60)

min_data = f_tickets.agg(F.min("Date_of_Purchase")).collect()[0][0]
max_data = f_tickets.agg(F.max("Date_of_Purchase")).collect()[0][0]
min_cal  = dim_calendario.agg(F.min("Date")).collect()[0][0]
max_cal  = dim_calendario.agg(F.max("Date")).collect()[0][0]

if min_cal <= min_data and max_cal >= max_data:
    print(f"✅ dim_calendario cobre {min_cal} → {max_cal}")
    print(f"   Fato vai de {min_data} → {max_data}")
else:
    erros.append(f"🔴 dim_calendario não cobre o range da fato!")
    print(erros[-1])

print()
print("=" * 60)
if len(erros) == 0:
    print("🎉 TODOS OS TESTES DAS DIMENSÕES PASSARAM!")
else:
    print(f"🔴 {len(erros)} TESTE(S) FALHARAM!")
    for e in erros: print(f"   {e}")
print("=" * 60)

DATA QUALITY CHECKS — DIMENSÕES
✅ dim_product: 42 registros
✅ dim_type: 5 registros
✅ dim_subject: 16 registros
✅ dim_status: 3 registros
✅ dim_priority: 4 registros
✅ dim_channel: 4 registros
✅ dim_customer: 8,320 registros
✅ dim_calendario: 730 registros

────────────────────────────────────────────────────────────
Integridade referencial — FKs x Dimensões
────────────────────────────────────────────────────────────
✅ Customer_ID → dim_customer: sem órfãos
✅ Product_ID → dim_product: sem órfãos
✅ Type_ID → dim_type: sem órfãos
✅ Subject_ID → dim_subject: sem órfãos
✅ Status_ID → dim_status: sem órfãos
✅ Priority_ID → dim_priority: sem órfãos
✅ Channel_ID → dim_channel: sem órfãos

────────────────────────────────────────────────────────────
Cobertura do calendário
────────────────────────────────────────────────────────────
✅ dim_calendario cobre 2020-01-01 → 2021-12-30
   Fato vai de 2020-01-01 → 2021-12-30

🎉 TODOS OS TESTES DAS DIMENSÕES PASSARAM!


In [0]:
### Célula 4 — Resumo executivo dos testes

In [0]:
print("=" * 60)
print("RESUMO EXECUTIVO — GOLD QUALITY CHECKS")
print("=" * 60)
print(f"""
TABELA FATO
  ✅ Volume:              8.469 registros
  ✅ Chave primária:      sem nulos, sem duplicatas
  ✅ Chaves estrangeiras: 7 FKs sem nulos
  ✅ Is_Resolved:         apenas 0 e 1
  ✅ Satisfaction:        range 1-5 respeitado
  ✅ Resolution_Time:     sem valores negativos
  ✅ Date_of_Purchase:    range 2020-2021 respeitado

DIMENSÕES
  ✅ dim_product:         42 produtos únicos
  ✅ dim_type:            5 tipos únicos
  ✅ dim_subject:         16 assuntos únicos
  ✅ dim_status:          3 status únicos
  ✅ dim_priority:        4 prioridades únicas
  ✅ dim_channel:         4 canais únicos
  ✅ dim_customer:        8.320 clientes únicos
  ✅ dim_calendario:      730 dias (2020-2021)

INTEGRIDADE REFERENCIAL
  ✅ Todos os 7 relacionamentos sem órfãos

COBERTURA TEMPORAL
  ✅ dim_calendario cobre 100% do range da fato
""")
print("=" * 60)
print("🎉 GOLD APROVADA — PIPELINE 100% ÍNTEGRO!")
print("=" * 60)

RESUMO EXECUTIVO — GOLD QUALITY CHECKS

TABELA FATO
  ✅ Volume:              8.469 registros
  ✅ Chave primária:      sem nulos, sem duplicatas
  ✅ Chaves estrangeiras: 7 FKs sem nulos
  ✅ Is_Resolved:         apenas 0 e 1
  ✅ Satisfaction:        range 1-5 respeitado
  ✅ Resolution_Time:     sem valores negativos
  ✅ Date_of_Purchase:    range 2020-2021 respeitado

DIMENSÕES
  ✅ dim_product:         42 produtos únicos
  ✅ dim_type:            5 tipos únicos
  ✅ dim_subject:         16 assuntos únicos
  ✅ dim_status:          3 status únicos
  ✅ dim_priority:        4 prioridades únicas
  ✅ dim_channel:         4 canais únicos
  ✅ dim_customer:        8.320 clientes únicos
  ✅ dim_calendario:      730 dias (2020-2021)

INTEGRIDADE REFERENCIAL
  ✅ Todos os 7 relacionamentos sem órfãos

COBERTURA TEMPORAL
  ✅ dim_calendario cobre 100% do range da fato

🎉 GOLD APROVADA — PIPELINE 100% ÍNTEGRO!
